# 09 — Retrain in the cloud from a local notebook (Azure ML)

**Question answered here:** *"To retrain, do I drive Azure ML from a local
notebook? How much longer does Azure ML take versus training directly on the
local PC (accounting for the compute-node start time — one node)?"*

**Short answer.** Yes — this notebook is the local driver. It submits the
training *command job* to the Azure ML workspace via the `azure-ai-ml` SDK,
streams it to completion, and downloads the ONNX artifacts back into
[`models/`](../models). It is a thin wrapper around
[`cloud-training/submit_job.py`](../cloud-training/submit_job.py) so there is a
single source of truth.

### Timing: local PC vs Azure ML (single node)

The *training compute itself* is essentially the same on both — these are tiny
CPU models (~160 k params, a few thousand windows), ~20–35 s per machine. The
difference is **fixed per-job overhead** that Azure ML adds and the local PC
does not:

| phase | local PC | Azure ML (cold, 1 node, scale-to-zero) | Azure ML (warm node) |
|---|---|---|---|
| queue + scheduling | 0 | ~10–30 s | ~10–30 s |
| **node allocation** (0→1 node) | 0 | **~2–5 min** | 0 |
| image / conda env pull | 0 (already local) | ~1–3 min first time, cached after | ~0–20 s |
| training (per machine) | ~25–35 s | ~25–35 s | ~25–35 s |
| artifact upload + download | 0 | ~10–30 s | ~10–30 s |

So on a **cold** cluster (our `cpu-cluster` is `min_nodes=0`, so it scales to
zero between runs) Azure ML adds roughly **3–8 minutes of overhead** before the
first epoch, dominated by allocating the single node and pulling the image the
first time. On a **warm** node (a second run within the idle window) the
overhead drops to **well under a minute**. The training time itself does not
improve over the local PC for a model this small — a CPU node is comparable to
a laptop CPU.

**Practical consequence (and why M-003 was trained locally):** for these small
models the cold-start overhead is *larger than the training itself*, so for a
single 3-sensor model like **M-003** we train locally with
[`tools/train_cnc_m003.py`](../tools/train_cnc_m003.py) (~10 min wall-clock
including data generation, no node wait). The cloud path pays off when you
(a) retrain the **whole fleet** in parallel, (b) need a **GPU**, or (c) want a
tracked, reproducible run in the workspace. The cell below *measures* the
actual per-phase time so you can see the overhead for your cluster state.

> **Secrets:** this notebook reads everything sensitive from `.env`. Azure ML
> resource *names* (subscription / resource group / workspace / compute) are
> not secrets, but you can still override them from `.env` (keys
> `AML_SUBSCRIPTION`, `AML_RESOURCE_GROUP`, `AML_WORKSPACE`, `AML_COMPUTE`).
> Auth uses your existing `az login` identity — no keys in code.

In [ ]:
from __future__ import annotations

import os
import sys
import time
from datetime import datetime, timezone
from pathlib import Path

from dotenv import load_dotenv

REPO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
load_dotenv(REPO / ".env")
sys.path.insert(0, str(REPO / "cloud-training"))
import submit_job as sj  # single source of truth for build_job/get_client/download_models

# Optional .env overrides for the (non-secret) AML config.
sj.SUBSCRIPTION = os.environ.get("AML_SUBSCRIPTION", sj.SUBSCRIPTION)
sj.RESOURCE_GROUP = os.environ.get("AML_RESOURCE_GROUP", sj.RESOURCE_GROUP)
sj.WORKSPACE = os.environ.get("AML_WORKSPACE", sj.WORKSPACE)
sj.COMPUTE = os.environ.get("AML_COMPUTE", sj.COMPUTE)
print("workspace :", sj.WORKSPACE)
print("resource  :", sj.RESOURCE_GROUP)
print("compute   :", sj.COMPUTE, "(min_nodes=0 -> scales to zero between runs)")

## Which machines to (re)train

M-001 / M-002 are the FSM physics machines. M-003 (CNC) is normally trained
locally, but you *can* include it in the cloud fleet job if you also add its
profile + generator to `cloud-training/src` — by default we keep the cloud job
to the FSM fleet.

In [ ]:
MACHINES = ["M-001", "M-002"]   # edit as needed
EPOCHS = 12
HOURS = 8.0                      # hours of synthetic telemetry per machine
sj.MACHINES = MACHINES
print("will train:", MACHINES, f"(epochs={EPOCHS}, hours={HOURS})")

In [ ]:
# Connect to the workspace (uses AzureCliCredential -> DefaultAzureCredential).
ml = sj.get_client()
print("connected to", ml.workspace_name)

In [ ]:
# Submit the command job and stream status, MEASURING per-phase wall-clock so
# the cold-start overhead is visible empirically.
job = ml.jobs.create_or_update(sj.build_job(EPOCHS, HOURS))
print("job name :", job.name)
print("studio   :", job.studio_url)

terminal = {"Completed", "Failed", "Canceled"}
t0 = time.time()
phase_start = {}
last = None
while True:
    j = ml.jobs.get(job.name)
    if j.status != last:
        now = time.time()
        if last is not None:
            print(f"  [{last:>12s}] took {now - phase_start[last]:6.1f} s")
        phase_start[j.status] = now
        print(f"[{datetime.now(timezone.utc).strftime('%H:%M:%S')}] -> {j.status}")
        last = j.status
    if j.status in terminal:
        break
    time.sleep(15)
print(f"\nTOTAL wall-clock: {time.time() - t0:6.1f} s  (final status: {last})")

In [ ]:
# Download the produced ONNX artifacts into models/ (AAD blob auth).
if last == "Completed":
    sj.download_models(ml, job.name)
    print("artifacts downloaded into models/")
else:
    print(f"job did not complete (status={last}); see studio link above")

### Reading the measured times

The `[Starting]` / `[Preparing]` phases above are the **node allocation +
image pull** — that is the time Azure ML adds that a local run does not have.
Subtract the `[Running]` phase (the actual training, comparable to local) from
the total to get the pure cloud overhead for your cluster's current warm/cold
state. For a cold `min_nodes=0` cluster expect the bulk of the wall-clock to
be in node allocation, confirming the table at the top of the notebook.